In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 5.99951171875 GB


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM,TrainingArguments,Trainer,BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

W1106 06:50:18.622000 47264 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1106 06:50:19.393000 47264 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [ ]:
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, ##quantizaiton in 4 btis
    bnb_4bit_quant_type="nf4",## ahndels rpecisison there ar 16 possible numebrs in 4 btis sicne bits are clsoer to 0 nf4 keeps the 16 valeus clsoes to 0
    bnb_4bit_compute_dtype=torch.bfloat16 ## 16 computations bits
    
)



model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto",
    trust_remote_code = True
)



tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          rust_remote_code=True)

Loading weights: 100%|██████████| 338/338 [00:05<00:00, 64.81it/s]


In [5]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[   "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)


model = get_peft_model(model, lora_config)

In [6]:
data = load_dataset(
    "json",
    data_files="train.jsonl",
    split="train"
)

In [11]:
def tokenize(batch):
    texts = []

    for instruction, out in zip(batch["instruction"], batch["response"]):
        text = f"Instruction: {instruction}\nResponse: {out}"
        texts.append(text)

    tokens = tokenizer(
    texts,
    padding=False,
    max_length=1024,
    truncation=True,
    ##return_tensors="pt" //dont want rectangualr shape
    )   

    tokens["labels"] = [
        ids.copy() for ids in tokens["input_ids"]
    ]

    return tokens


In [16]:
from transformers import DataCollatorForSeq2Seq

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

In [17]:
tokenized_data = data.map(
    tokenize,
    batched=True,
    remove_columns=data.column_names
)

In [18]:
training_args = TrainingArguments(
    output_dir = '../models/tinyllama-lora-tuned',
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate = 1e-3,
    num_train_epochs = 3,
    fp16 = True,
    logging_steps = 20,
    save_strategy = 'epoch',
    report_to = 'none',
    remove_unused_columns = False,      
    label_names = ["labels"]
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_data,
    processing_class = tokenizer,
    data_collator=collator
)

In [19]:
import torch

print("Allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved:", torch.cuda.memory_reserved() / 1024**3, "GB")
print("Total:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

Allocated: 1.1388225555419922 GB
Reserved: 1.1796875 GB
Total: 5.99951171875 GB


In [20]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Step,Training Loss
20,0.540838
40,0.118400
60,0.070024
80,0.053901
100,0.041485
120,0.040414
140,0.035204
160,0.033661
180,0.031534
200,0.028690


TrainOutput(global_step=279, training_loss=0.07897625217301017, metrics={'train_runtime': 1454.206, 'train_samples_per_second': 0.763, 'train_steps_per_second': 0.192, 'total_flos': 2222106318397440.0, 'train_loss': 0.07897625217301017, 'epoch': 3.0})

In [21]:
print("Training examples:", len(trainer.train_dataset))
print("Batch size:", trainer.args.per_device_train_batch_size)
print("Gradient accumulation:", trainer.args.gradient_accumulation_steps)
print("Total steps:", trainer.state.global_step)

Training examples: 370
Batch size: 1
Gradient accumulation: 4
Total steps: 279


In [23]:
model.save_pretrained("../models/qwen-lora-tuned-sql")
tokenizer.save_pretrained("../models/qwen-lora-tuned-sql")

('./qwen-lora-tuned-sql\\tokenizer_config.json',
 './qwen-lora-tuned-sql\\chat_template.jinja',
 './qwen-lora-tuned-sql\\tokenizer.json')

In [ ]:
import torch



raw_data =  load_dataset(
    "json",
    data_files="test.jsonl",
    split="train"
)

refs = raw_data["response"]

def generate_sql(model, instruction):
    model.eval()

    prompt = f"Instruction: {instruction}\nResponse:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # Extract only newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()